## PS1 - rescue robot

In [1]:
import sys, time
from collections import deque

MOVES = [("UP", -1, 0), ("RIGHT", 0, 1), ("DOWN", 1, 0), ("LEFT", 0, -1)]

In [2]:
def in_bounds(r, c, R, C):
    return 0 <= r < R and 0 <= c < C

def build_path(parent, node):
    directions = []
    while node in parent:
        prev, d = parent[node]
        directions.append(d)
        node = prev
    directions.reverse()
    return directions

### bfs

In [3]:
def bfs(grid, R, C, start, goal):
    t0 = time.time()
    visited = {start}
    parent = {}
    queue = deque([start])
    nodes_expanded = 0
    while queue:
        cur = queue.popleft()
        nodes_expanded += 1
        if cur == goal:
            return True, build_path(parent, cur), nodes_expanded, time.time() - t0
        r, c = cur
        for name, dr, dc in MOVES:
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc, R, C) and grid[nr][nc] != "#" and (nr, nc) not in visited:
                visited.add((nr, nc))
                parent[(nr, nc)] = (cur, name)
                queue.append((nr, nc))
    return False, [], nodes_expanded, time.time() - t0

In [4]:
toy = ['S.', '.G']
print(bfs(toy, 2, 2, (0,0), (1,1)))

(True, ['RIGHT', 'DOWN'], 4, 0.0)


### dfs

In [5]:
def dfs(grid, R, C, start, goal):
    t0 = time.time()
    visited = {start}
    parent = {}
    stack = [start]
    nodes_expanded = 0
    while stack:
        cur = stack.pop()
        nodes_expanded += 1
        if cur == goal:
            return True, build_path(parent, cur), nodes_expanded, time.time() - t0
        r, c = cur
        for name, dr, dc in reversed(MOVES):  # reversed so UP still pops first
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc, R, C) and grid[nr][nc] != "#" and (nr, nc) not in visited:
                visited.add((nr, nc))
                parent[(nr, nc)] = (cur, name)
                stack.append((nr, nc))
    return False, [], nodes_expanded, time.time() - t0

In [6]:
print(dfs(toy, 2, 2, (0,0), (1,1)))

(True, ['RIGHT', 'DOWN'], 3, 0.0)


In [7]:
# wall maze, first try sealed the corners off completely (no path)
maze = ['S.#', '.#.', '#..']
print(bfs(maze, 3, 3, (0,0), (2,2)))

(False, [], 3, 0.0)


In [8]:
maze = ['S..', '.#.', '..G']  # single center wall instead
print(bfs(maze, 3, 3, (0,0), (2,2)))
print(dfs(maze, 3, 3, (0,0), (2,2)))

(True, ['RIGHT', 'RIGHT', 'DOWN', 'DOWN'], 8, 0.0)
(True, ['RIGHT', 'RIGHT', 'DOWN', 'DOWN'], 5, 0.0)


### the real grid

In [9]:
def parse_grid(text):
    lines = text.split(chr(10))
    r, c = map(int, lines[0].split())
    grid = lines[1:1+r]
    start = goal = None
    for i in range(r):
        for j in range(c):
            if grid[i][j] == 'S':
                start = (i, j)
            elif grid[i][j] == 'G':
                goal = (i, j)
    return grid, r, c, start, goal

sample1 = '''5 5\nS..#.\n#..#.\n.....\n.##..\n....G'''
grid, R, C, start, goal = parse_grid(sample1)
print(start, goal)  # manhattan dist 8, no obvious detour needed

(0, 0) (4, 4)


In [10]:
print(bfs(grid, R, C, start, goal))

(True, ['RIGHT', 'RIGHT', 'DOWN', 'DOWN', 'RIGHT', 'RIGHT', 'DOWN', 'DOWN'], 19, 0.0)


In [11]:
print(dfs(grid, R, C, start, goal))  # same path here, grid barely branches

(True, ['RIGHT', 'RIGHT', 'DOWN', 'DOWN', 'RIGHT', 'RIGHT', 'DOWN', 'DOWN'], 11, 0.0)


In [12]:
sample2 = '''5 5\nS....\n####.\n....#\n.####\n....G'''
grid2, R2, C2, start2, goal2 = parse_grid(sample2)
print(bfs(grid2, R2, C2, start2, goal2))
print(dfs(grid2, R2, C2, start2, goal2))

(False, [], 6, 0.0)
(False, [], 6, 0.0)


### full run, matches the required output format

In [13]:
def print_result(name, found, path, nodes, t):
    print(f'Algorithm: {name}')
    if not found:
        print('Path Found: No')
        print(f'Nodes Expanded = {nodes}')
        print(f'Execution Time = {t:.6f}')
        return
    print('Path Found: Yes')
    print('Path: ' + ' '.join(path))
    print(f'Number of Moves = {len(path)}')
    print(f'Nodes Expanded = {nodes}')
    print(f'Execution Time = {t:.6f}')

In [14]:
import io
sys.stdin = io.StringIO(sample1)
g, R, C, s, gl = parse_grid(sys.stdin.read())

bf, bp, bn, bt = bfs(g, R, C, s, gl)
print_result('BFS', bf, bp, bn, bt)
print()
df, dp, dn, dt = dfs(g, R, C, s, gl)
print_result('DFS', df, dp, dn, dt)
print()
print('Comparison:')
print(f'Path length -> BFS = {len(bp)}, DFS = {len(dp)}')
print(f'Nodes expanded -> BFS = {bn}, DFS = {dn}')
print('BFS is optimal since it explores level by level, DFS is not.')

Algorithm: BFS
Path Found: Yes
Path: RIGHT RIGHT DOWN DOWN RIGHT RIGHT DOWN DOWN
Number of Moves = 8
Nodes Expanded = 19
Execution Time = 0.000000

Algorithm: DFS
Path Found: Yes
Path: RIGHT RIGHT DOWN DOWN RIGHT RIGHT DOWN DOWN
Number of Moves = 8
Nodes Expanded = 11
Execution Time = 0.000000

Comparison:
Path length -> BFS = 8, DFS = 8
Nodes expanded -> BFS = 19, DFS = 11
BFS is optimal since it explores level by level, DFS is not.
